In [1]:
import keras
from keras.layers import Dense, Conv2D, BatchNormalization, Activation
from keras.layers import AveragePooling2D, Input, Flatten
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint, LearningRateScheduler
from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.regularizers import l2
from keras import backend as K
from keras.models import Model
import numpy as np
import os

In [2]:
import pandas as pd
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical

def load_and_preprocess_data(csv_path, image_base_path, angles_to_process=None):
    # Load the metadata CSV
    df = pd.read_csv(csv_path)
    angle_groups = df.groupby('cam_angle')
    
    # If angles_to_process is not specified, process all angles
    if angles_to_process is None:
        angles_to_process = angle_groups.groups.keys()
    else:
        angles_to_process = [angles_to_process] if isinstance(angles_to_process, (int, float)) else angles_to_process
    
    X_all, y_all = [], []
    
    # Iterate through each angle and process
    for angle in angles_to_process:
        if angle not in angle_groups.groups:
            print(f"Angle {angle} not found in the dataset. Skipping.")
            continue
        print(f"Processing angle: {angle}")
        group = angle_groups.get_group(angle)
        
        for _, row in group.iterrows():
            image_path = os.path.join(image_base_path, row['image_filename'])
            try:
                img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    img = cv2.resize(img, (240, 240))
                    img = img.astype('float32') / 255.0  # Normalize to [0, 1]
                    X_all.append(img)
                    y_all.append(row['label'])
                else:
                    print(f"Failed to load image: {image_path}")
            except Exception as e:
                print(f"Error processing image {image_path}: {str(e)}")
    
    if not X_all:
        raise ValueError("No valid images found in the dataset.")
    
    X_all = np.array(X_all)
    y_all = np.array(y_all)
    
    # Use LabelEncoder to ensure labels are zero-indexed
    label_encoder = LabelEncoder()
    y_all = label_encoder.fit_transform(y_all)
    
    # Split the data into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42)
    
    # Reshape the data to match the CIFAR-10 format
    X_train = X_train.reshape(-1, 240, 240, 1)
    X_test = X_test.reshape(-1, 240, 240, 1)
    
    # Get the number of unique classes
    num_classes = len(label_encoder.classes_)
    
    return (X_train, y_train), (X_test, y_test), num_classes

def load_data():
    # Define the paths
    csv_path = '/kaggle/input/processed-geni2/geni_metadata.csv'
    image_base_path = '/kaggle/input/processed-geni2/geni_images'
    
    # Specify the angles to process
    angles_to_process = [0, 18, 36, 54, 72, 90, 108, 126, 144, 162, 180]
    
    # Load and preprocess the data
    return load_and_preprocess_data(csv_path, image_base_path, angles_to_process)

# Usage example:
(X_train, y_train), (X_test, y_test), num_classes = load_data()

# Print summary of the loaded data
print("\nSummary of loaded data:")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Image shape: {X_train.shape[1:]}")
print(f"Number of classes: {num_classes}")

# Data Preprocessing
subtract_pixel_mean = True

# If subtract pixel mean is enabled
if subtract_pixel_mean:
    x_train_mean = np.mean(X_train, axis=0)
    X_train -= x_train_mean
    X_test -= x_train_mean

# Convert class vectors to binary class matrices.
y_train = keras.utils.to_categorical(y_train, num_classes) 
y_test = keras.utils.to_categorical(y_test, num_classes)

# Print shapes after preprocessing
print("\nAfter preprocessing:")
print('X_train shape:', X_train.shape)
print(X_train.shape[0], 'train samples')
print(X_test.shape[0], 'test samples')
print('y_train shape:', y_train.shape)

Processing angle: 0
Processing angle: 18
Processing angle: 36
Processing angle: 54
Processing angle: 72
Processing angle: 90
Processing angle: 108
Processing angle: 126
Processing angle: 144
Processing angle: 162
Processing angle: 180

Summary of loaded data:
Training samples: 6524
Testing samples: 1631
Image shape: (240, 240, 1)
Number of classes: 124

After preprocessing:
X_train shape: (6524, 240, 240, 1)
6524 train samples
1631 test samples
y_train shape: (6524, 124)


In [3]:
# Setting LR for different number of Epochs 
def lr_schedule(epoch): 
	lr = 1e-3
	if epoch > 180: 
		lr *= 0.5e-3
	elif epoch > 160: 
		lr *= 1e-3
	elif epoch > 120: 
		lr *= 1e-2
	elif epoch > 80: 
		lr *= 1e-1
	print('Learning rate: ', lr) 
	return lr 


In [4]:
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation
from tensorflow.keras.regularizers import l2

def resnet_layer(inputs,
                 num_filters=16,
                 kernel_size=3,
                 strides=1,
                 activation='relu',
                 batch_normalization=True,
                 conv_first=True):
    """2D Convolution-Batch Normalization-Activation stack builder

    Args:
        inputs (tensor): input tensor from input image or previous layer
        num_filters (int): Conv2D number of filters
        kernel_size (int): Conv2D square kernel dimensions
        strides (int): Conv2D square stride dimensions
        activation (string): activation name
        batch_normalization (bool): whether to include batch normalization
        conv_first (bool): conv-bn-activation (True) or bn-activation-conv (False)

    Returns:
        x (tensor): tensor as input to the next layer
    """
    conv = Conv2D(num_filters,
                  kernel_size=kernel_size,
                  strides=strides,
                  padding='same',
                  kernel_initializer='he_normal',
                  kernel_regularizer=l2(1e-4))

    x = inputs
    if conv_first:
        x = conv(x)
        if batch_normalization:
            x = BatchNormalization()(x)
        if activation is not None:
            x = Activation(activation)(x)
    else:
        if batch_normalization:
            x = BatchNormalization()(x)
        if activation is not None:
            x = Activation(activation)(x)
        x = conv(x)
    return x

In [5]:
from tensorflow import keras
from tensorflow.keras.layers import Input, Activation, AveragePooling2D, Flatten, Dense
from tensorflow.keras.models import Model

def resnet_v1(input_shape, depth, num_classes=10):
    if (depth - 2) % 6 != 0:
        raise ValueError('depth should be 6n + 2 (eg 20, 32, 44 in [a])')
    
    # Start model definition.
    num_filters = 16
    num_res_blocks = int((depth - 2) / 6)

    inputs = Input(shape=input_shape)
    x = resnet_layer(inputs=inputs)
    
    # Instantiate the stack of residual units
    for stack in range(3):
        for res_block in range(num_res_blocks):
            strides = 1
            if stack > 0 and res_block == 0:  # first layer but not first stack
                strides = 2  # downsample
            y = resnet_layer(inputs=x,
                             num_filters=num_filters,
                             strides=strides)
            y = resnet_layer(inputs=y,
                             num_filters=num_filters,
                             activation=None)
            if stack > 0 and res_block == 0:  # first layer but not first stack
                # linear projection residual shortcut connection to match changed dims
                x = resnet_layer(inputs=x,
                                 num_filters=num_filters,
                                 kernel_size=1,
                                 strides=strides,
                                 activation=None,
                                 batch_normalization=False)
            x = keras.layers.add([x, y])
            x = Activation('relu')(x)
        num_filters *= 2

    # Add classifier on top.
    # v1 does not use BN after last shortcut connection-ReLU
    x = AveragePooling2D(pool_size=8)(x)
    y = Flatten()(x)
    outputs = Dense(num_classes,
                    activation='softmax',
                    kernel_initializer='he_normal')(y)

    # Instantiate model.
    model = Model(inputs=inputs, outputs=outputs)
    return model

In [6]:
from tensorflow import keras
from tensorflow.keras.layers import Input, Activation, AveragePooling2D, Flatten, Dense
from tensorflow.keras.models import Model

def resnet_v1(input_shape, depth, num_classes=124):  # Changed default to 124
    if (depth - 2) % 6 != 0:
        raise ValueError('depth should be 6n + 2 (eg 20, 32, 44 in [a])')
    
    # Start model definition.
    num_filters = 16
    num_res_blocks = int((depth - 2) / 6)

    inputs = Input(shape=input_shape)
    x = resnet_layer(inputs=inputs)
    
    # Instantiate the stack of residual units
    for stack in range(3):
        for res_block in range(num_res_blocks):
            strides = 1
            if stack > 0 and res_block == 0:  # first layer but not first stack
                strides = 2  # downsample
            y = resnet_layer(inputs=x,
                             num_filters=num_filters,
                             strides=strides)
            y = resnet_layer(inputs=y,
                             num_filters=num_filters,
                             activation=None)
            if stack > 0 and res_block == 0:  # first layer but not first stack
                # linear projection residual shortcut connection to match changed dims
                x = resnet_layer(inputs=x,
                                 num_filters=num_filters,
                                 kernel_size=1,
                                 strides=strides,
                                 activation=None,
                                 batch_normalization=False)
            x = keras.layers.add([x, y])
            x = Activation('relu')(x)
        num_filters *= 2

    # Add classifier on top.
    # v1 does not use BN after last shortcut connection-ReLU
    x = AveragePooling2D(pool_size=8)(x)
    y = Flatten()(x)
    outputs = Dense(num_classes,
                    activation='softmax',
                    kernel_initializer='he_normal')(y)

    # Instantiate model.
    model = Model(inputs=inputs, outputs=outputs)
    return model

# Create the model
input_shape = X_train.shape[1:]  # Assuming x_train is your input data
depth = 20  # or whatever depth you're using
model = resnet_v1(input_shape=input_shape, depth=depth, num_classes=124)

# Print model summary
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 240, 240,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 240, 240,  │        160 │ input_layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 240, 240,  │         64 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 240, 240,  │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 240, 240,  │      2,320 │ activation[0][0]  │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 240, 240,  │         64 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 240, 240,  │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 240, 240,  │      2,320 │ activation_1[0][… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 240, 240,  │         64 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 240, 240,  │          0 │ activation[0][0], │
│                     │ 16)               │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 240, 240,  │          0 │ add[0][0]         │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 240, 240,  │      2,320 │ activation_2[0][… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 240, 240,  │         64 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 240, 240,  │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 240, 240,  │      2,320 │ activation_3[0][… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 240, 240,  │         64 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 240, 240,  │          0 │ activation_2[0][

 Total params: 662,492 (2.53 MB)

 Trainable params: 661,116 (2.52 MB)

 Non-trainable params: 1,376 (5.38 KB)

In [7]:
from tensorflow import keras
from tensorflow.keras.callbacks import ModelCheckpoint, LearningRateScheduler, ReduceLROnPlateau

# Learning rate schedule
def lr_schedule(epoch):
    lr = 1e-3
    if epoch > 180:
        lr *= 0.5e-3
    elif epoch > 160:
        lr *= 1e-3
    elif epoch > 120:
        lr *= 1e-2
    elif epoch > 80:
        lr *= 1e-1
    print('Learning rate: ', lr)
    return lr

# Set up model name and save directory
version = 1  # Assuming you're using version 1
depth = 20  # Assuming you're using depth 20
model_type = f'ResNet{depth}v{version}'
model_name = f'msi_{model_type}_model.keras'
save_dir = os.path.join(os.getcwd(), 'saved_models')
if not os.path.isdir(save_dir):
    os.makedirs(save_dir)
filepath = os.path.join(save_dir, model_name)

# Prepare callbacks for model saving and for learning rate adjustment
checkpoint = ModelCheckpoint(
    filepath=filepath,
    monitor='val_accuracy',
    verbose=1,
    save_best_only=True)

lr_scheduler = LearningRateScheduler(lr_schedule)

lr_reducer = ReduceLROnPlateau(
    factor=np.sqrt(0.1),
    cooldown=0,
    patience=5,
    min_lr=0.5e-6)

callbacks = [checkpoint, lr_reducer, lr_scheduler]

# Compile model
model.compile(loss='categorical_crossentropy',
              optimizer=keras.optimizers.Adam(learning_rate=lr_schedule(0)),
              metrics=['accuracy'])

# Run training
batch_size = 32  # Adjust as needed
epochs = 100     # Adjust as needed

history = model.fit(X_train, y_train,
                    batch_size=batch_size,
                    epochs=epochs,
                    validation_data=(X_test, y_test),
                    shuffle=True,
                    callbacks=callbacks)

Learning rate:  0.001
Learning rate:  0.001
Epoch 1/100


I0000 00:00:1728475658.481272      76 service.cc:145] XLA service 0x7d31a0001350 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1728475658.481337      76 service.cc:153]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1728475658.481342      76 service.cc:153]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1728475675.753307      76 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


204/204 ━━━━━━━━━━━━━━━━━━━━ 0s 381ms/step - accuracy: 0.0107 - loss: 6.4412
Epoch 1: val_accuracy improved from -inf to 0.00858, saving model to /kaggle/working/saved_models/msi_ResNet20v1_model.keras
204/204 ━━━━━━━━━━━━━━━━━━━━ 117s 431ms/step - accuracy: 0.0107 - loss: 6.4361 - val_accuracy: 0.0086 - val_loss: 5.0103 - learning_rate: 0.0010
Learning rate:  0.001
Epoch 2/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 0s 338ms/step - accuracy: 0.0863 - loss: 4.2709
Epoch 2: val_accuracy improved from 0.00858 to 0.08032, saving model to /kaggle/working/saved_models/msi_ResNet20v1_model.keras
204/204 ━━━━━━━━━━━━━━━━━━━━ 73s 358ms/step - accuracy: 0.0864 - loss: 4.2700 - val_accuracy: 0.0803 - val_loss: 4.3517 - learning_rate: 0.0010
Learning rate:  0.001
Epoch 3/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 0s 339ms/step - accuracy: 0.2364 - loss: 3.2178
Epoch 3: val_accuracy improved from 0.08032 to 0.23360, saving model to /kaggle/working/saved_models/msi_ResNet20v1_model.keras
204/204 ━━━━━━━━━━━━━━━━━━━━ 73

In [8]:
# Score trained model. 
scores = model.evaluate(X_test, y_test, verbose = 1) 
print('Test loss:', scores[0]) 
print('Test accuracy:', scores[1])

51/51 ━━━━━━━━━━━━━━━━━━━━ 4s 79ms/step - accuracy: 0.8020 - loss: 1.3237
Test loss: 1.218456506729126
Test accuracy: 0.8025751113891602
